In [0]:
print("Hello")

Hello


In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_source (
session_id INT,
zone_id STRING,
vehicle_type STRING,
duration_hours INT,
parking_fee INT,
status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_sessions_source VALUES
(101, 'Z01', 'Car', 3, 150, 'Completed'),
(102, 'Z02', 'Bike', 2, 40, 'Completed'),
(103, 'Z03', 'Car', 5, 250, 'Completed'),
(104, 'Z01', 'SUV', 4, 240, 'Active'),
(105, 'Z04', 'Bike', 1, 20, 'Completed'),
(106, 'Z02', 'Car', 6, 300, 'Completed'),
(107, 'Z03', 'SUV', 2, 120, 'Cancelled'),
(108, 'Z05', 'Car', 3, 150, 'Completed'),
(109, 'Z01', 'Bike', 4, 80, 'Completed'),
(110, 'Z04', 'SUV', 5, 300, 'Active'),
(111, 'Z02', 'Car', 2, 100, 'Completed'),
(112, 'Z05', 'Bike', 3, 60, 'Cancelled'),
(113, 'Z03', 'Car', 7, 350, 'Completed'),
(114, 'Z01', 'SUV', 3, 180, 'Completed'),
(115, 'Z04', 'Bike', 2, 40, 'Active'),
(116, 'Z05', 'Car', 4, 200, 'Completed'),
(117, 'Z02', 'SUV', 6, 360, 'Completed'),
(118, 'Z03', 'Bike', 1, 20, 'Completed'),
(119, 'Z04', 'Car', 5, 250, 'Completed'),
(120, 'Z05', 'SUV', 2, 120, 'Active');

num_affected_rows,num_inserted_rows
20,20


In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_work
USING DELTA
AS
SELECT * FROM parking_sessions_source;


num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE parking_zones (
zone_id STRING,
zone_name STRING,
city STRING,
capacity INT
)
USING DELTA;
INSERT INTO parking_zones VALUES
('Z01', 'Central Plaza', 'Hyderabad', 100),
('Z02', 'Metro Parking', 'Bengaluru', 150),
('Z03', 'City Mall', 'Mumbai', 200),
('Z04', 'Airport Parking', 'Delhi', 250),
('Z05', 'Tech Park', 'Pune', 120);

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parking_sessions AS
SELECT *
FROM VALUES
(102, 'Z02', 'Bike', 3, 60, 'Completed'),
(104, 'Z01', 'SUV', 5, 300, 'Completed'),
(121, 'Z03', 'Car', 4, 200, 'Completed'),
(122, 'Z05', 'Bike', 2, 40, 'Active')
AS t(
session_id,
zone_id,
vehicle_type,
duration_hours,
parking_fee,
status
);

In [0]:
%sql
select * from parking_sessions_source;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed


In [0]:
%sql 
select session_id,vehicle_type,parking_fee from parking_sessions_source;

session_id,vehicle_type,parking_fee
101,Car,150
102,Bike,40
103,Car,250
104,SUV,240
105,Bike,20
106,Car,300
107,SUV,120
108,Car,150
109,Bike,80
110,SUV,300


In [0]:
%sql
select * from parking_sessions_source 
where status="Completed" and parking_fee > 200;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
103,Z03,Car,5,250,Completed
106,Z02,Car,6,300,Completed
113,Z03,Car,7,350,Completed
117,Z02,SUV,6,360,Completed
119,Z04,Car,5,250,Completed


In [0]:
%sql
select vehicle_type,
sum(parking_fee) as Total_fee,
count(session_id) as Count 
from parking_sessions_source
group by vehicle_type;

vehicle_type,Total_fee,Count
Car,1750,8
Bike,260,6
SUV,1320,6


In [0]:
from pyspark.sql.functions import col,when
df=spark.table("parking_sessions_source")
res_df=df.withColumn(
    "fee_category",
    when(col("parking_fee")>200,"High")
    .otherwise("Low")

)
res_df.show()

+----------+-------+------------+--------------+-----------+---------+------------+
|session_id|zone_id|vehicle_type|duration_hours|parking_fee|   status|fee_category|
+----------+-------+------------+--------------+-----------+---------+------------+
|       101|    Z01|         Car|             3|        150|Completed|         Low|
|       102|    Z02|        Bike|             2|         40|Completed|         Low|
|       103|    Z03|         Car|             5|        250|Completed|        High|
|       104|    Z01|         SUV|             4|        240|   Active|        High|
|       105|    Z04|        Bike|             1|         20|Completed|         Low|
|       106|    Z02|         Car|             6|        300|Completed|        High|
|       107|    Z03|         SUV|             2|        120|Cancelled|         Low|
|       108|    Z05|         Car|             3|        150|Completed|         Low|
|       109|    Z01|        Bike|             4|         80|Completed|      

In [0]:
%sql
CREATE TABLE parking_summary
USING DELTA
AS
SELECT *
FROM parking_sessions_source
WHERE status = 'Completed';

num_affected_rows,num_inserted_rows


In [0]:
%sql
insert into parking_sessions_source values
(123,"Z01","Car",3,150,"Active");

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
update parking_sessions_source
set status="Completed",parking_fee=280
where session_id=104;

num_affected_rows
1


In [0]:
%sql
delete from parking_sessions_source where status="Cancelled";
select * from parking_sessions_source;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
104,Z01,SUV,4,280,Completed
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parking_sessions AS
SELECT *
FROM VALUES
(102, 'Z02', 'Bike', 3, 60, 'Completed'),
(104, 'Z01', 'SUV', 5, 300, 'Completed'),
(121, 'Z03', 'Car', 4, 200, 'Completed'),
(122, 'Z05', 'Bike', 2, 40, 'Active')
AS t(
session_id,
zone_id,
vehicle_type,
duration_hours,
parking_fee,
status
);

In [0]:
%sql
select * from incoming_parking_sessions;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed
104,Z01,SUV,5,300,Completed
121,Z03,Car,4,200,Completed
122,Z05,Bike,2,40,Active


In [0]:
%sql
select p.session_id from parking_sessions_source p
inner join incoming_parking_sessions i
on p.session_id=i.session_id;

session_id
102
104


In [0]:
%sql
MERGE INTO parking_sessions_source AS target
USING incoming_parking_sessions AS source
ON target.session_id = source.session_id

WHEN MATCHED THEN
  UPDATE SET
    target.vehicle_type = source.vehicle_type,
    target.parking_fee = source.parking_fee,
    target.duration_hours = source.duration_hours,
    target.status = source.status

WHEN NOT MATCHED THEN
  INSERT (
    session_id,
    zone_id,
    vehicle_type,
    duration_hours,
    parking_fee,
    status
  )
  VALUES (
    source.session_id,
    source.zone_id,
    source.vehicle_type,
    source.duration_hours,
    source.parking_fee,
    source.status
  );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,4,0,0


In [0]:
%sql
describe history parking_sessions_source;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
9,2026-11-06T06:10:15.000Z,70638158589839,kishorekarthick0803@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1822430682077403),b88ce661-40a9-40af-8c96-f7dc73c7d0bc,1106-045943-af3qxpj9-v2n,8,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 13696, p25FileSize -> 3135, numDeletionVectorsRemoved -> 1, minFileSize -> 3135, numAddedFiles -> 1, maxFileSize -> 3135, p75FileSize -> 3135, p50FileSize -> 3135, numAddedBytes -> 3135)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
8,2026-11-06T06:10:13.000Z,70638158589839,kishorekarthick0803@gmail.com,MERGE,"Map(predicate -> [""(session_id#14034 = session_id#14016)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1822430682077403),b88ce661-40a9-40af-8c96-f7dc73c7d0bc,1106-045943-af3qxpj9-v2n,7,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 10561, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 4, executionTimeMs -> 3176, materializeSourceTimeMs -> 114, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 923, numTargetRowsUpdated -> 4, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2094)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
7,2026-11-06T06:09:45.000Z,70638158589839,kishorekarthick0803@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1822430682077403),2bad6053-d18f-4e80-8c3a-36cd0eb34dc5,1106-045943-af3qxpj9-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 13611, p25FileSize -> 3135, numDeletionVectorsRemoved -> 1, minFileSize -> 3135, numAddedFiles -> 1, maxFileSize -> 3135, p75FileSize -> 3135, p50FileSize -> 3135, numAddedBytes -> 3135)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
6,2026-11-06T06:09:43.000Z,70638158589839,kishorekarthick0803@gmail.com,MERGE,"Map(predicate -> [""(session_id#13241 = session_id#13223)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1822430682077403),2bad6053-d18f-4e80-8c3a-36cd0eb34dc5,1106-045943-af3qxpj9-v2n,5,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 10479, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3989, materializeSourceTimeMs -> 280, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1308, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2325)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-06T05:19:26.000Z,70638158589839,kishorekarthick0803@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1822430682077403),aa40cbb2-1e34-4abf-9e28-5989f0598b7b,1106-045943-af3qxpj9-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6780, p2

In [0]:
%sql
SELECT
    'BEFORE MERGE' AS record_state,
    *
FROM parking_sessions_source VERSION AS OF 5
WHERE session_id = 102

UNION ALL

SELECT
    'AFTER MERGE' AS record_state,
    *
FROM parking_sessions_source
WHERE session_id = 102;

record_state,session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
BEFORE MERGE,102,Z02,Bike,2,40,Completed
AFTER MERGE,102,Z02,Bike,3,60,Completed


In [0]:
%sql
CREATE VIEW completed_parking_view AS
SELECT *
FROM parking_sessions_source
WHERE status = 'Completed';

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW zone_z01_sessions AS
SELECT *
FROM parking_sessions_source
WHERE zone_id = 'Z01';

In [0]:
%sql
CREATE OR REPLACE GLOBAL TEMP VIEW high_fee_sessions AS
SELECT *
FROM parking_sessions_source
WHERE parking_fee > 200;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6072029434668416>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE OR REPLACE GLOBAL TEMP VIEW high_fee_sessions AS\nSELECT *\nFROM parking_sessions_source\nWHERE parking_fee > 200;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:217, in SqlMagic.sql(self, lin

In [0]:
%sql
SHOW VIEWS;

namespace,viewName,isTemporary,isMaterialized,isMetric
default,completed_parking_view,false,false,false
,incoming_parking_sessions,true,false,false
,zone_z01_sessions,true,false,false


In [0]:
%sql
describe detail parking_sessions_source;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,2cd8ec95-04a2-42be-9da2-38c90a18f424,workspace.default.parking_sessions_source,null,,2026-11-06T04:23:24.859Z,2026-11-06T06:10:15.000Z,List(),List(),1,3135,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-7761e6e7-977c-40ce-8125-1fb1494a0611, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-03950ce0-d42a-4f94-9a34-0c425052503f)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
optimize parking_sessions_source;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793946856398, 1793946857341, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, null, null, 0)"


In [0]:
%sql
describe detail parking_sessions_source;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,2cd8ec95-04a2-42be-9da2-38c90a18f424,workspace.default.parking_sessions_source,null,,2026-11-06T04:23:24.859Z,2026-11-06T06:10:15.000Z,List(),List(),1,3135,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-7761e6e7-977c-40ce-8125-1fb1494a0611, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-03950ce0-d42a-4f94-9a34-0c425052503f)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


What OPTIMIZE does

Compacts multiple small Parquet files into fewer larger files.
Reduces file-opening overhead.
Improves query performance, especially for large Delta tables.
Does not change the data itself.

Why a small dataset may show little or no change

The table may already consist of one or very few files.
There may be insufficient data to create larger files.
Delta Lake may determine that further compaction provides no performance benefit.
The dataset size may be much smaller than the target file size used by OPTIMIZE.

In [0]:
from pyspark.sql import SparkSession
rate_df = (
    spark.readStream
         .format("rate")
         .option("rowsPerSecond", 5)
         .load()
)

display(rate_df)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6072029434668424>, line 9
      1 from pyspark.sql import SparkSession
      2 rate_df = (
      3     spark.readStream
      4          .format("rate")
      5          .option("rowsPerSecond", 5)
      6          .load()
      7 )
----> 9 display(rate_df)

File <command-6072029434668424>, line 3
      1 from pyspark.sql import SparkSession
      2 rate_df = (
----> 3     spark.readStream
      4          .format("rate")
      5          .option("rowsPerSecond", 5)
      6          .load()
      7 )
      9 display(rate_df)

AnalysisException: [TEMP_CHECKPOINT_LOCATION_NOT_SUPPORTED] Implicit temporary streaming checkpoint locations are not supported in the current workspace, please specify a checkpoint location explicitly.
For display(), set the checkpoint location using:
display(df, checkpointLocation = "your_path")
For

In [0]:
from pyspark.sql.functions import col
rate_df = (
    spark.readStream
         .format("rate")
         .option("rowsPerSecond", 5)
         .load()
)
filtered_df = (
    rate_df
    .withColumn("amount", col("value") * 10)
    .filter(col("amount") > 300)
)
filtered_df.show()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8638103587686885>, line 13
      2 rate_df = (
      3     spark.readStream
      4          .format("rate")
      5          .option("rowsPerSecond", 5)
      6          .load()
      7 )
      8 filtered_df = (
      9     rate_df
     10     .withColumn("amount", col("value") * 10)
     11     .filter(col("amount") > 300)
     12 )
---> 13 filtered_df.show()

File <command-8638103587686885>, line 9
      1 from pyspark.sql.functions import col
      2 rate_df = (
      3     spark.readStream
      4          .format("rate")
      5          .option("rowsPerSecond", 5)
      6          .load()
      7 )
      8 filtered_df = (
----> 9     rate_df
     10     .withColumn("amount", col("value") * 10)
     11     .filter(col("amount") > 300)
     12 )
     13 filtered_df.show()

File <command-8638103587686885>, line 3
     

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE bronze_parking
AS
SELECT *
FROM STREAM(parking_sessions_source);

CREATE OR REFRESH STREAMING TABLE silver_completed_parking
AS
SELECT *
FROM STREAM(bronze_parking)
WHERE session_status = 'Completed';

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8638103587686890>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "CREATE OR REFRESH STREAMING TABLE bronze_parking\nAS\nSELECT *\nFROM STREAM(parking_sessions_source);\n\nCREATE OR REFRESH STREAMING TABLE silver_completed_parking\nAS\nSELECT *\nFROM STREAM(bronze_parking)\nWHERE session_status = 'Completed';\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_C

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW gold_zone_summary
AS
SELECT
    zone_id,
    COUNT(session_id) AS total_completed_sessions,
    SUM(parking_fee) AS total_parking_revenue
FROM silver_completed_parking
GROUP BY zone_id;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8638103587686899>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE OR REFRESH MATERIALIZED VIEW gold_zone_summary\nAS\nSELECT\n    zone_id,\n    COUNT(session_id) AS total_completed_sessions,\n    SUM(parking_fee) AS total_parking_revenue\nFROM silver_completed_parking\nGROUP BY zone_id;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED,